In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report)

#from google.colab import drive

#drive.mount('/content/drive')

#df = pd.read_excel('/content/drive/MyDrive/ML/Telco-Customer-Churn.xlsx')

df = pd.read_excel('Telco-Customer-Churn.xlsx')


In [2]:
print("Original shape:", df.shape)
print(df.head())
print(df.columns)

Original shape: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV Streaming

In [3]:
df.columns = df.columns.str.strip()

In [4]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [5]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [6]:
X = df.drop(['customerID', 'Churn'], axis=1)
y = df['Churn']

In [7]:
X = pd.get_dummies(X, drop_first=True)

In [8]:
print("Final X shape:", X.shape)
print("Final y shape:", y.shape)

Final X shape: (7043, 30)
Final y shape: (7043,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (5282, 30)
X_test shape: (1761, 30)


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Feature scaling applied (StandardScaler).")

Feature scaling applied (StandardScaler).


In [11]:
from sklearn.model_selection import cross_val_score

# 5-fold CV on Logistic Regression
cv_scores_lr = cross_val_score(
    LogisticRegression(max_iter=2000), X_train, y_train,
    cv=5, scoring='roc_auc'
)
print("LR 5-Fold CV AUC: %.4f (+/- %.4f)" % (cv_scores_lr.mean(), cv_scores_lr.std()))

# 5-fold CV on Random Forest
cv_scores_rf = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train,
    cv=5, scoring='roc_auc'
)
print("RF 5-Fold CV AUC: %.4f (+/- %.4f)" % (cv_scores_rf.mean(), cv_scores_rf.std()))

LR 5-Fold CV AUC: 0.8439 (+/- 0.0131)
RF 5-Fold CV AUC: 0.8239 (+/- 0.0130)


In [12]:
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("AUC:", roc_auc_score(y_test, y_prob_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression
Accuracy: 0.8063600227143668
AUC: 0.8463870474500992
Confusion Matrix:
 [[1161  133]
 [ 208  259]]
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1294
           1       0.66      0.55      0.60       467

    accuracy                           0.81      1761
   macro avg       0.75      0.73      0.74      1761
weighted avg       0.80      0.81      0.80      1761



In [13]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("AUC:", roc_auc_score(y_test, y_prob_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest
Accuracy: 0.7887563884156729
AUC: 0.826310032467425
Confusion Matrix:
 [[1160  134]
 [ 238  229]]
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1294
           1       0.63      0.49      0.55       467

    accuracy                           0.79      1761
   macro avg       0.73      0.69      0.71      1761
weighted avg       0.78      0.79      0.78      1761



In [14]:
import joblib

# Save trained models and scaler
joblib.dump(lr, 'lr_model.pkl')
joblib.dump(rf, 'rf_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("Models and scaler saved successfully.")

Models and scaler saved successfully.
